# End-to-End Incremental Employee Load

Incremental medallion pipeline over the employee CSVs in `/Volumes/workspace/config/employee`: **bronze** ingests only new files via Auto Loader (`trigger(availableNow=True)`), **silver** merges typed, deduplicated rows per employee, and **gold** rebuilds department aggregates.


In [0]:
# Cell 1 — BRONZE: incremental (Auto Loader) ingest of CSV files from the volume
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/workspace/config/employee"
BRONZE_TABLE = "workspace.bronze.employee_bronze"
CHECKPOINT_BRONZE = "/Volumes/workspace/config/employee/_checkpoints/bronze"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE}
(
  empno      STRING,
  ename      STRING,
  dept       STRING,
  salary     STRING,
  createdat  STRING,
  modifiedat STRING,
  _source_file STRING,
  _ingested_at TIMESTAMP
)
""")

# Auto Loader processes only NEW files on each run (incremental)
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", CHECKPOINT_BRONZE)
    .option("header", "true")
    .load(SOURCE_PATH)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", CHECKPOINT_BRONZE)
    .trigger(availableNow=True)   # run once, process new files, then stop
    .toTable(BRONZE_TABLE)
)

print("Bronze incremental load complete.")


In [0]:
# Cell 2 — SILVER: typed, deduplicated current state (MERGE keeps it incremental & idempotent)
from pyspark.sql.window import Window

SILVER_TABLE = "workspace.silver.employee_silver"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
(
  empno      INT,
  ename      STRING,
  dept       INT,
  salary     DOUBLE,
  createdat  TIMESTAMP,
  modifiedat TIMESTAMP
)
""")

# Cast bronze strings to proper types and keep only the newest row per employee
latest_per_employee = (
    spark.read.table(BRONZE_TABLE)
    .select(
        F.col("empno").cast("INT").alias("empno"),
        F.col("ename"),
        F.col("dept").cast("INT").alias("dept"),
        F.col("salary").cast("DOUBLE").alias("salary"),
        F.col("createdat").cast("TIMESTAMP").alias("createdat"),
        F.col("modifiedat").cast("TIMESTAMP").alias("modifiedat"),
    )
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("empno").orderBy(F.col("modifiedat").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

latest_per_employee.createOrReplaceTempView("updates")

spark.sql(f"""
MERGE INTO {SILVER_TABLE} AS tgt
USING updates AS src
  ON tgt.empno = src.empno
WHEN MATCHED AND src.modifiedat > tgt.modifiedat THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print("Silver incremental merge complete.")


In [0]:
# Cell 2b — SILVER: SCD Type 2 (full history) to complement the SCD Type 1 table above.
#
# Cell 2 already implements SCD Type 1: the MERGE keeps only the CURRENT state
# of each employee (older versions are overwritten).
# This cell keeps the FULL VERSION HISTORY of every employee using the
# __START_AT / __END_AT convention:
#   __END_AT IS NULL  -> the currently active version
#   __END_AT = X      -> version superseded at time X
# (On DBR 17.3+ with a Pro/Serverless SQL warehouse you could instead use a
#  CREATE OR REFRESH STREAMING TABLE ... FLOW AUTO CDC ... STORED AS SCD TYPE 2
#  directly on the append-only bronze table, which handles all of this for you.)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SCD2_TABLE = "workspace.silver.employee_silver_scd2"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCD2_TABLE}
(
  empno      INT,
  ename      STRING,
  dept       INT,
  salary     DOUBLE,
  createdat  TIMESTAMP,
  modifiedat TIMESTAMP,
  __START_AT TIMESTAMP,  -- version valid from (= modifiedat of that version)
  __END_AT   TIMESTAMP   -- NULL = current version, otherwise when it was superseded
)
""")

# Latest incoming version per employee (same dedup logic as the Type 1 merge)
updates_scd2 = (
    spark.read.table(BRONZE_TABLE)
    .select(
        F.col("empno").cast("INT").alias("empno"),
        F.col("ename"),
        F.col("dept").cast("INT").alias("dept"),
        F.col("salary").cast("DOUBLE").alias("salary"),
        F.col("createdat").cast("TIMESTAMP").alias("createdat"),
        F.col("modifiedat").cast("TIMESTAMP").alias("modifiedat"),
    )
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("empno").orderBy(F.col("modifiedat").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("__START_AT", F.col("modifiedat"))
    .withColumn("__END_AT", F.lit(None).cast("TIMESTAMP"))
)
updates_scd2.createOrReplaceTempView("updates_scd2")

# Step 1 — close the currently open version when a newer one arrives
# (idempotent: only rows with __END_AT IS NULL can match, and once closed they no longer do)
spark.sql(f"""
MERGE INTO {SCD2_TABLE} AS tgt
USING updates_scd2 AS src
  ON tgt.empno = src.empno AND tgt.__END_AT IS NULL
WHEN MATCHED AND src.modifiedat > tgt.modifiedat
  THEN UPDATE SET tgt.__END_AT = src.__START_AT
""")

# Step 2 — insert versions that are not in the history yet and are not older
# than an already-stored version (the anti-join makes reruns idempotent)
spark.sql(f"""
INSERT INTO {SCD2_TABLE}
SELECT src.*
FROM updates_scd2 AS src
WHERE NOT EXISTS (
  SELECT 1 FROM {SCD2_TABLE} AS t
  WHERE t.empno = src.empno AND t.__START_AT >= src.__START_AT
)
""")

print("Silver SCD Type 2 history merge complete.")

# Current state derived from the SCD2 table — should match the Type 1 table above
print("\nCurrent versions (SCD Type 1 view over the SCD Type 2 history):")
display(spark.sql(f"""
  SELECT * EXCEPT (__START_AT, __END_AT)
  FROM {SCD2_TABLE}
  WHERE __END_AT IS NULL
  ORDER BY empno
"""))

print("\nFull version history per employee:")
display(spark.read.table(SCD2_TABLE).orderBy("empno", "__START_AT"))


In [0]:
# Cell 2c — SCD Type 1 and SCD Type 2 in pure PySpark (DataFrame + DeltaTable API)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE = "workspace.bronze.employee_bronze"
SILVER_TABLE = "workspace.silver.employee_silver"      # SCD Type 1 (current state)
SCD2_TABLE   = "workspace.silver.employee_silver_scd2"  # SCD Type 2 (full history)

# Both tables already exist from the cells above; recreate only if missing
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
( empno INT, ename STRING, dept INT, salary DOUBLE,
  createdat TIMESTAMP, modifiedat TIMESTAMP )
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCD2_TABLE}
( empno INT, ename STRING, dept INT, salary DOUBLE,
  createdat TIMESTAMP, modifiedat TIMESTAMP,
  __START_AT TIMESTAMP, __END_AT TIMESTAMP )
""")

# 1) Typed projection of the raw bronze rows
updates = (
    spark.read.table(BRONZE_TABLE)
    .select(
        F.col("empno").cast("INT").alias("empno"),
        F.col("ename"),
        F.col("dept").cast("INT").alias("dept"),
        F.col("salary").cast("DOUBLE").alias("salary"),
        F.col("createdat").cast("TIMESTAMP").alias("createdat"),
        F.col("modifiedat").cast("TIMESTAMP").alias("modifiedat"),
    )
)

# 2) Deduplicate: keep only the newest version per employee
latest = (
    updates
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("empno").orderBy(F.col("modifiedat").desc())))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ---------------- SCD TYPE 1: upsert the current state ----------------
# Same row per employee, overwritten when a newer version arrives.
delta_t1 = DeltaTable.forName(spark, SILVER_TABLE)
(delta_t1.alias("tgt")
    .merge(latest.alias("src"), "tgt.empno = src.empno")
    .whenMatchedUpdateAll(condition="src.modifiedat > tgt.modifiedat")
    .whenNotMatchedInsertAll()
    .execute())
print("SCD Type 1 (current state) merge complete.")

# ---------------- SCD TYPE 2: keep the full version history ----------------
# Convention: __END_AT IS NULL -> active version, __END_AT = X -> superseded at X.
scd2_updates = (
    latest
    .withColumn("__START_AT", F.col("modifiedat"))
    .withColumn("__END_AT", F.lit(None).cast("TIMESTAMP"))
)

# Step A: close the currently open version when a newer one arrives
# (idempotent: only rows with __END_AT IS NULL can match)
delta_t2 = DeltaTable.forName(spark, SCD2_TABLE)
(delta_t2.alias("tgt")
    .merge(scd2_updates.alias("src"),
           "tgt.empno = src.empno AND tgt.__END_AT IS NULL")
    .whenMatchedUpdate(
        condition="src.modifiedat > tgt.modifiedat",
        set={"__END_AT": "src.__START_AT"})
    .execute())

# Step B: insert versions not yet present (anti-join makes reruns idempotent)
existing = spark.read.table(SCD2_TABLE)
new_versions = scd2_updates.join(
    existing,
    (existing["empno"] == scd2_updates["empno"])
    & (existing["__START_AT"] >= scd2_updates["__START_AT"]),
    "left_anti",
)
new_versions.write.mode("append").saveAsTable(SCD2_TABLE)
print("SCD Type 2 (full history) merge complete.")

# Quick check: the SCD2 current rows should match the Type 1 table
print("\nSCD Type 1 — current state:")
display(spark.read.table(SILVER_TABLE).orderBy("empno"))

print("\nSCD Type 2 — full version history:")
display(spark.read.table(SCD2_TABLE).orderBy("empno", "__START_AT"))


In [0]:
# Cell 3 — GOLD: department-level aggregates rebuilt from silver after each incremental run
GOLD_TABLE = "workspace.gold.employee_gold"

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_TABLE} AS
SELECT
  dept,
  COUNT(*)              AS headcount,
  ROUND(AVG(salary), 2) AS avg_salary,
  MIN(salary)           AS min_salary,
  MAX(salary)           AS max_salary,
  SUM(salary)           AS total_payroll
FROM {SILVER_TABLE}
GROUP BY dept
ORDER BY dept
""")

print("Gold aggregation complete.")
display(spark.read.table(GOLD_TABLE))


In [0]:
# Cell 4 — Sanity checks across all three layers
print("Bronze rows (raw ingest):      ", spark.read.table(BRONZE_TABLE).count())
print("Silver rows (1 per employee):  ", spark.read.table(SILVER_TABLE).count())
print("Gold rows (1 per department):  ", spark.read.table(GOLD_TABLE).count())

display(spark.read.table(SILVER_TABLE).orderBy("empno"))
